# NumPy Dojo — Block 5: Core NumPy Operations

Covers the functions that come up constantly but are easy to misuse or forget:  
`pad`, `cumsum`, `diff`, `percentile`, `bincount`, `unique`, `argsort`, `searchsorted`, `roll`, `meshgrid`, `einsum` patterns, and the log-sum-exp trick.

Dataset: **sklearn digits** + synthetic sequences and distributions.  
Each problem forces the right function to emerge from the task — not listed upfront.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(42)

digits = load_digits()
images = digits.images   # (1797, 8, 8)
labels = digits.target   # (1797,)

# Synthetic time series: temperature readings over 100 days
time_series = 20 + 5 * np.sin(np.linspace(0, 4 * np.pi, 100)) + np.random.randn(100)

# Batch of logits (no softmax applied)
logits = np.random.randn(64, 10) * 3.0

print(f'images: {images.shape}, labels: {labels.shape}')
print(f'time_series: {time_series.shape}')

---
## P1 — Padding

You need to pad the digit images to prepare them for a convolution with a 3×3 kernel (valid conv shrinks the image — padding preserves size).

Tasks:
1. Add **zero-padding** of width 1 to all spatial dims of the image batch `(N, H, W)` → `(N, H+2, W+2)`
2. Add **reflect-padding** of width 2 to a single image `(H, W)` → `(H+4, W+4)`
3. Add **constant-padding** with value 255 to the right and bottom only (asymmetric)

In [ ]:
batch = images[:8]  # (8, 8, 8)
single = images[0]  # (8, 8)

def zero_pad_batch(imgs, pad):
    """imgs: (N,H,W), pad: int — pad all spatial dims equally"""
    # TODO: np.pad — only pad H and W, not N
    # Hint: pad_width is a list of (before, after) tuples, one per axis
    pass

def reflect_pad(img, pad):
    """img: (H,W), pad: int"""
    # TODO: mode='reflect'
    pass

def asymmetric_pad(img, bottom, right, value=255):
    """img: (H,W), pads only bottom and right"""
    # TODO: mode='constant', pad_width uses 0 for top/left
    pass

padded_batch  = zero_pad_batch(batch, pad=1)
padded_ref    = reflect_pad(single, pad=2)
padded_asym   = asymmetric_pad(single, bottom=3, right=3)

In [ ]:
# --- ASSERTS ---
N, H, W = batch.shape
assert padded_batch.shape == (N, H + 2, W + 2), f'Got {padded_batch.shape}'
# Corners must be zero
assert padded_batch[0, 0, 0] == 0 and padded_batch[0, -1, -1] == 0
# Original content preserved
assert np.allclose(padded_batch[:, 1:-1, 1:-1], batch)

assert padded_ref.shape == (H + 4, W + 4)
# Reflect: padded_ref[0, pad] should equal padded_ref[2*pad, pad] (mirror)
assert padded_ref[0, 2] == padded_ref[4, 2], 'Reflect padding wrong'

assert padded_asym.shape == (H + 3, W + 3)
assert np.all(padded_asym[-3:, :] == 255), 'Bottom padding value wrong'
assert np.all(padded_asym[:, -3:] == 255), 'Right padding value wrong'
assert np.all(padded_asym[:H, :W] == single), 'Original content changed'

print('P1 PASSED ✓')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(single, cmap='gray'); axes[0].set_title('Original (8×8)')
axes[1].imshow(padded_batch[0], cmap='gray'); axes[1].set_title('Zero-padded (10×10)')
axes[2].imshow(padded_ref, cmap='gray'); axes[2].set_title('Reflect-padded (12×12)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## P2 — cumsum and diff

Using the `time_series` temperature data:

1. Compute the **7-day running total** of temperatures (cumulative sum, then subtract the lagged version)
2. Compute **day-to-day changes** (first discrete difference)
3. Find all days where the temperature **crossed above 22°C** (was below on previous day, above on current)

In [ ]:
def rolling_sum(x, window):
    """
    x: (N,), window: int
    Returns: (N - window + 1,) — rolling sum using cumsum
    The trick: cumsum[i] - cumsum[i-window] = sum of window elements ending at i
    """
    # TODO: compute cumsum, then use slicing to get rolling sums
    # Don't use a loop
    pass

def daily_changes(x):
    """First-order difference: x[i] - x[i-1]"""
    # TODO: np.diff
    pass

def crossing_days(x, threshold):
    """
    Returns indices where x crosses above threshold
    i.e. x[i-1] < threshold and x[i] >= threshold
    """
    # TODO: boolean arrays + np.where or np.diff trick
    pass

rolling = rolling_sum(time_series, window=7)
changes = daily_changes(time_series)
crossings = crossing_days(time_series, threshold=22.0)

In [ ]:
# --- ASSERTS ---
assert rolling.shape == (len(time_series) - 7 + 1,), f'Rolling shape: {rolling.shape}'
# First element should equal sum of first 7 days
assert np.isclose(rolling[0], time_series[:7].sum(), atol=1e-6)
assert np.isclose(rolling[1], time_series[1:8].sum(), atol=1e-6)

assert changes.shape == (len(time_series) - 1,)
assert np.isclose(changes[0], time_series[1] - time_series[0])

# Verify crossings
for i in crossings:
    assert time_series[i] >= 22.0, f'Day {i} not above threshold'
    assert time_series[i-1] < 22.0, f'Day {i-1} should be below threshold'

print(f'P2 PASSED ✓  crossings at days: {crossings}')

---
## P3 — percentile and robust normalization

Standard z-score normalization is sensitive to outliers. A robust alternative uses median and IQR.

1. Implement **robust scaling**: `(X - median) / IQR` where IQR = 75th - 25th percentile, per feature
2. Compare with z-score on a dataset with injected outliers — show robustness visually
3. Find pixels that are **above the 90th percentile** in at least one image (hot pixel detection)

In [ ]:
X = digits.data.astype(float)  # (1797, 64)

# Inject outliers into a copy
X_outlier = X.copy()
X_outlier[np.random.choice(len(X), 20), np.random.choice(64, 20)] = 1000.0

def robust_scale(X):
    """
    X: (N, D)
    Returns: X_scaled (N, D) — per-feature median/IQR normalization
    Guard against IQR=0 (constant features)
    """
    # TODO: np.percentile with axis=0
    # IQR = 75th percentile - 25th percentile
    pass

def zscore(X):
    mu = X.mean(axis=0); s = X.std(axis=0); s[s==0] = 1
    return (X - mu) / s

def hot_pixels(imgs, percentile=90):
    """
    imgs: (N, H, W)
    Returns (row, col) indices of pixels that exceed the per-image 90th percentile
    in at least one image — shape: two arrays of int
    """
    # TODO: compute per-image threshold, build mask across all images, find spatial positions
    pass

X_robust = robust_scale(X_outlier)
X_z      = zscore(X_outlier)
hot_r, hot_c = hot_pixels(images)

In [ ]:
# --- ASSERTS ---
assert X_robust.shape == X.shape
assert not np.any(np.isnan(X_robust))

# Robust: median of scaled should be near 0, even with outliers
clean_feat_median = np.median(X_robust[:, X.std(axis=0) > 0], axis=0)
assert np.abs(clean_feat_median).mean() < 0.1, 'Robust median should be near 0'

# Robustness comparison: robust should have smaller max than zscore
assert X_robust.max() < X_z.max(), 'Robust scaling should suppress outliers more than z-score'

assert hot_r.shape == hot_c.shape
assert np.all(hot_r >= 0) and np.all(hot_r < 8)
assert np.all(hot_c >= 0) and np.all(hot_c < 8)

print(f'P3 PASSED ✓  hot pixels found: {len(hot_r)}')
print(f'  z-score max with outliers: {X_z.max():.1f}')
print(f'  robust max with outliers:  {X_robust.max():.1f}')

---
## P4 — bincount and unique

1. Count class frequencies in `labels` using `np.bincount` — no loops, no Counter
2. Compute **class weights** for imbalanced training: `weight[c] = N / (num_classes * count[c])`
3. Find **unique predicted sequences**: given predicted labels from 5 models (each predicts 1797 labels), find which predictions are identical across models
4. Compute the **mode** per feature in a matrix — most frequent value per column

In [ ]:
# Simulate imbalanced labels
np.random.seed(0)
imbalanced = np.random.choice(10, size=1000, p=[0.3,0.2,0.15,0.1,0.07,0.06,0.05,0.04,0.02,0.01])

# 5 model predictions
preds_5models = np.stack([np.random.randint(0, 10, size=1797) for _ in range(5)])  # (5, 1797)

def class_counts(y, num_classes):
    """Returns (num_classes,) count array"""
    # TODO: np.bincount — handle the case where some classes may not appear
    pass

def class_weights(y, num_classes):
    """Returns (num_classes,) weight array — higher weight for rarer classes"""
    # TODO: use class_counts, then formula: N / (num_classes * count)
    pass

def unique_predictions(preds):
    """
    preds: (M, N) — M models, N samples
    Returns number of unique prediction vectors (rows)
    """
    # TODO: np.unique with axis=0
    pass

def column_mode(X):
    """
    X: (N, D) integer array
    Returns (D,) — most frequent value per column
    """
    # TODO: np.unique with return_counts=True per column, or np.bincount per column
    # Try to do it without a Python for loop (or with minimal looping)
    pass

counts = class_counts(imbalanced, num_classes=10)
weights = class_weights(imbalanced, num_classes=10)
n_unique = unique_predictions(preds_5models)
int_images = (images * 2).astype(int)  # make integer for mode
modes = column_mode(digits.data.astype(int).reshape(1797, 64))

In [ ]:
# --- ASSERTS ---
assert counts.shape == (10,)
assert counts.sum() == 1000
assert counts[0] > counts[9], 'Class 0 should be most frequent'

assert weights.shape == (10,)
# Rarer class (9) should have higher weight than common (0)
assert weights[9] > weights[0], 'Rarer class should have higher weight'
# Weighted sum of counts * weights should equal num_classes * something consistent
assert np.isclose((counts * weights).sum(), len(imbalanced), atol=1.0)

assert isinstance(n_unique, (int, np.integer))
assert 1 <= n_unique <= 5

assert modes.shape == (64,)
print(f'P4 PASSED ✓  unique predictions across 5 models: {n_unique}')
print(f'  Class counts (imbalanced): {counts}')

---
## P5 — argsort and searchsorted

1. **Top-k predictions**: given softmax probabilities `(N, C)`, return top-3 class indices per sample — shape `(N, 3)`, sorted by descending probability
2. **Ranking**: given model scores, compute the rank of each sample (1 = highest score)
3. **Bucket assignment**: given continuous values and bin edges, assign each value to a bucket index using `searchsorted` — no loops

In [ ]:
probs = np.abs(np.random.randn(32, 10))
probs /= probs.sum(axis=1, keepdims=True)

scores = np.random.randn(100)

values = np.random.uniform(0, 100, size=200)
bin_edges = np.array([0, 10, 25, 50, 75, 90, 100], dtype=float)

def top_k(probs, k):
    """
    probs: (N, C)
    Returns: (N, k) — indices of top-k classes per sample, descending order
    """
    # TODO: np.argsort descending, then slice first k columns
    # argsort returns ascending — think about how to reverse
    pass

def rank_samples(scores):
    """
    scores: (N,)
    Returns: (N,) integer ranks where 1 = highest score
    """
    # TODO: np.argsort twice (argsort of argsort = rank)
    pass

def bucket_assign(values, bin_edges):
    """
    Assign each value in `values` to a bucket index [0, num_bins)
    bin_edges[i] <= value < bin_edges[i+1] → bucket i
    """
    # TODO: np.searchsorted — read the 'side' parameter docs
    pass

topk = top_k(probs, k=3)
ranks = rank_samples(scores)
buckets = bucket_assign(values, bin_edges)

In [ ]:
# --- ASSERTS ---
assert topk.shape == (32, 3)
# Verify: prob at top1 index >= prob at top2 index, for all samples
for i in range(32):
    assert probs[i, topk[i, 0]] >= probs[i, topk[i, 1]] >= probs[i, topk[i, 2]], f'Sample {i} top-k not sorted'

assert ranks.shape == (100,)
assert ranks.min() == 1 and ranks.max() == 100
# Highest score should have rank 1
assert ranks[scores.argmax()] == 1, 'Highest score should be rank 1'
assert ranks[scores.argmin()] == 100, 'Lowest score should be rank 100'

assert buckets.shape == (200,)
# All bucket indices should be valid
num_bins = len(bin_edges) - 1
assert np.all(buckets >= 0) and np.all(buckets < num_bins), f'Bucket out of range: {buckets.min()}, {buckets.max()}'
# Value of 0 should be in bucket 0, value of 99 should be in bucket 5
assert bucket_assign(np.array([0.0]), bin_edges)[0] == 0
assert bucket_assign(np.array([99.0]), bin_edges)[0] == 5

print('P5 PASSED ✓')

---
## P6 — roll and meshgrid

1. **Circular shift**: shift a 1D signal by `k` positions (positive = right). Use `np.roll`. Then implement the **same thing without `np.roll`** using only indexing.
2. **Sliding window average** on an image using roll — a crude blur
3. **Meshgrid**: create a 2D grid of coordinates for an H×W image, compute the L2 distance from the center for every pixel (creates a radial distance map)

In [ ]:
signal = np.array([1., 2., 3., 4., 5., 6., 7., 8.])
img = images[0]  # (8, 8)

def circular_shift_roll(x, k):
    """Shift right by k positions using np.roll"""
    # TODO
    pass

def circular_shift_index(x, k):
    """Same thing using only indexing — no np.roll"""
    # TODO: think about what rolling by k means in terms of index modulo N
    pass

def blur_with_roll(img, radius=1):
    """
    img: (H, W)
    Average of (2*radius+1)^2 neighbours — crude box blur using roll
    Returns same shape
    """
    # TODO: sum shifted versions of img (±1 in each direction), divide by count
    # Use np.roll with axis=0 and axis=1
    pass

def radial_distance_map(H, W):
    """
    Returns (H, W) array where each element is L2 distance from image center
    """
    # TODO: np.meshgrid to get (x, y) coordinate grids
    # Center = (H//2, W//2)
    pass

shifted_roll  = circular_shift_roll(signal, k=3)
shifted_index = circular_shift_index(signal, k=3)
blurred = blur_with_roll(img)
dist_map = radial_distance_map(8, 8)

In [ ]:
# --- ASSERTS ---
expected_shift = np.array([6., 7., 8., 1., 2., 3., 4., 5.])
assert np.allclose(shifted_roll, expected_shift), f'roll shift wrong: {shifted_roll}'
assert np.allclose(shifted_index, expected_shift), f'index shift wrong: {shifted_index}'

assert blurred.shape == img.shape
# Blur should reduce range compared to original
assert blurred.max() - blurred.min() <= img.max() - img.min()

assert dist_map.shape == (8, 8)
assert dist_map[4, 4] == 0.0, 'Center should have distance 0'
# Corner (0,0) should have max distance
assert dist_map[0, 0] == dist_map.max()

print('P6 PASSED ✓')

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(blurred, cmap='gray'); axes[1].set_title('Roll-blurred')
axes[2].imshow(dist_map, cmap='plasma'); axes[2].set_title('Radial distance map')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## P7 — einsum fluency

Write an `np.einsum` string for each operation — **do not use any other function** (no `@`, no `np.dot`, no `np.sum`).

The subscript is the answer — figure it out from the description.

In [ ]:
A = np.random.randn(4, 5)    # (4, 5)
B = np.random.randn(5, 6)    # (5, 6)
v = np.random.randn(5)       # (5,)
C = np.random.randn(4, 5)    # same shape as A
T = np.random.randn(3, 4, 5) # (batch, N, M)
S = np.random.randn(3, 5, 5) # (batch, N, N) — square

# 1. Matrix-vector product: (4,5) x (5,) -> (4,)
res1 = np.einsum('TODO', A, v)

# 2. Matrix multiplication: (4,5) x (5,6) -> (4,6)
res2 = np.einsum('TODO', A, B)

# 3. Row-wise dot products: A[i,:] · C[i,:] for all i -> (4,)
res3 = np.einsum('TODO', A, C)

# 4. Outer product: v (5,) and v (5,) -> (5,5)
res4 = np.einsum('TODO', v, v)

# 5. A @ B.T using einsum: (4,5) x (4,5).T -> (4,4)
#    i.e. dot each row of A with each row of C
res5 = np.einsum('TODO', A, C)

# 6. Batch trace: extract diagonal sum from each (5,5) matrix in S -> (3,)
res6 = np.einsum('TODO', S)

# 7. Batch matmul: (3,4,5) x (3,5,6) via einsum -> (3,4,6)
Bmat = np.random.randn(3, 5, 6)
res7 = np.einsum('TODO', T, Bmat)

# 8. Element-wise multiply then sum last axis: (3,4,5) and (3,4,5) -> (3,4)
T2 = np.random.randn(3, 4, 5)
res8 = np.einsum('TODO', T, T2)

In [ ]:
# --- ASSERTS ---
assert res1.shape == (4,)  and np.allclose(res1, A @ v)
assert res2.shape == (4,6) and np.allclose(res2, A @ B)
assert res3.shape == (4,)  and np.allclose(res3, (A * C).sum(axis=1))
assert res4.shape == (5,5) and np.allclose(res4, np.outer(v, v))
assert res5.shape == (4,4) and np.allclose(res5, A @ C.T)
assert res6.shape == (3,)  and np.allclose(res6, np.array([np.trace(S[i]) for i in range(3)]))
assert res7.shape == (3,4,6)
assert res8.shape == (3,4)  and np.allclose(res8, (T * T2).sum(axis=-1))
print('P7 PASSED ✓  all einsum operations correct')

---
## P8 — Log-sum-exp trick

The log-sum-exp function: `log(∑ exp(x_i))` overflows for large x and underflows for very negative x.

1. Implement the **naive** version (direct exp then log) — show where it breaks
2. Implement the **stable** version using the identity: `LSE(x) = max(x) + log(∑ exp(x_i - max(x)))`
3. Implement **log-softmax** using LSE — more numerically stable than `log(softmax(x))`
4. Implement **NLL loss** using log-softmax (avoids `log(0)` issues)

This is the pattern behind all stable probability computations in deep learning.

In [ ]:
def logsumexp_naive(x):
    """x: (C,) or (N,C) — naive, will overflow"""
    # TODO: just np.log(np.exp(x).sum(axis=-1))
    pass

def logsumexp_stable(x):
    """
    x: (C,) or (N, C)
    Returns scalar or (N,)
    """
    # TODO: subtract max first
    pass

def log_softmax(x):
    """
    x: (N, C)
    Returns (N, C) — log of softmax probabilities
    More stable than log(softmax(x))
    """
    # TODO: x - logsumexp_stable(x) [broadcast correctly]
    pass

def nll_loss(logits, labels):
    """
    Negative log-likelihood using log_softmax
    logits: (N, C), labels: (N,)
    Returns scalar
    """
    # TODO: log_softmax, then pick correct class, negate and mean
    pass

In [ ]:
# --- ASSERTS ---

# Big values: stable should work, naive should produce inf or nan
big = np.array([1000., 1001., 1002.])
naive_out = logsumexp_naive(big)
stable_out = logsumexp_stable(big)
assert np.isnan(naive_out) or np.isinf(naive_out), 'Naive should overflow on [1000,1001,1002]'
print(f'  naive:  {naive_out}  ← expected inf/nan')
print(f'  stable: {stable_out:.6f}  ← expected ~1002.408')
assert np.isclose(stable_out, np.log(np.exp(0) + np.exp(1) + np.exp(2)) + 1000, atol=1e-4)

# Log-softmax: should equal log of regular softmax on normal values
x = logits[:8]
from_stable = log_softmax(x)
def softmax(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)
from_naive = np.log(softmax(x))
assert np.allclose(from_stable, from_naive, atol=1e-5), 'log_softmax mismatch'

# Log-softmax rows should sum to 0 in log space... actually check exp sums to 1
assert np.allclose(np.exp(from_stable).sum(axis=1), 1.0, atol=1e-5)

# NLL loss: same result as cross-entropy
y = np.random.randint(0, 10, 64)
nll = nll_loss(logits, y)
ce  = -np.log(softmax(logits)[np.arange(64), y]).mean()
assert np.isclose(nll, ce, atol=1e-5), f'NLL {nll:.4f} != CE {ce:.4f}'

print('\nP8 PASSED ✓')